In [6]:
# TeachingMonster Batch Runner (Google Colab)
#
# This notebook runs /content/TeachingMonster-released against every valid row
# in the first sheet (all topics) of topics.xlsx.
#
# Expected inputs in Colab:
# - TeachingMonster-released.zip or an already extracted /content/TeachingMonster-released
# - topics.xlsx
#
# Outputs are written to /content/teachingmonster_batch_outputs/<timestamp>/,
# with one subfolder per Request ID.

In [7]:
# 1) Install system dependencies required by TeachingMonster.
# Colab runtime may ask you to restart after package installation; if so, restart and rerun from this cell.

!apt-get update -qq
!apt-get install -y -qq libreoffice poppler-utils ffmpeg

print("System dependencies installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.11_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.11_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unselected package libuno-sal3.
Preparing to unpack .../002-libuno-sal3_1%3a7.3.7-0ubuntu0.22.04.11_amd64.deb ...
Unpacking libuno-sal3 (1:7.3.7-0ubuntu0.22.04.11) ...
Selecting previously unsele

In [8]:
# 2) Prepare project and spreadsheet in Colab without using files.upload().
# Recommended: put these two files in Google Drive first:
#   MyDrive/education_video/TeachingMonster-released.zip
#   MyDrive/education_video/topics.xlsx
#
# If you already uploaded/extracted them under /content, this cell will reuse them.

from pathlib import Path
import os
import shutil
import zipfile

content_dir = Path("/content")
project_dir = content_dir / "TeachingMonster-released"
topics_path = content_dir / "topics.xlsx"
drive_dir = Path("/content/drive/MyDrive/education_video")
project_zip = drive_dir / "TeachingMonster-released.zip"
drive_topics = drive_dir / "topics.xlsx"

os.chdir(content_dir)

# Mount Google Drive if the expected files are not already in /content.
if not project_dir.exists() or not topics_path.exists():
    from google.colab import drive
    drive.mount("/content/drive")

# Copy topics.xlsx from Drive when needed.
if not topics_path.exists() and drive_topics.exists():
    shutil.copy2(drive_topics, topics_path)

# Extract project zip from Drive when needed.
if not project_dir.exists() and project_zip.exists():
    with zipfile.ZipFile(project_zip, "r") as zf:
        zf.extractall(content_dir)
    # Handle zips that contain either TeachingMonster-released/ or a single top-level project folder.
    if not project_dir.exists():
        candidates = [p for p in content_dir.iterdir() if p.is_dir() and p.name.startswith("TeachingMonster")]
        if candidates:
            candidates[0].rename(project_dir)

if not project_dir.exists():
    raise FileNotFoundError(
        "Missing project. Put TeachingMonster-released.zip at "
        "/content/drive/MyDrive/education_video/TeachingMonster-released.zip "
        "or extract the folder to /content/TeachingMonster-released."
    )
if not topics_path.exists():
    raise FileNotFoundError(
        "Missing topics.xlsx. Put it at "
        "/content/drive/MyDrive/education_video/topics.xlsx "
        "or upload it to /content/topics.xlsx."
    )

print("Project:", project_dir)
print("Topics:", topics_path)

Mounted at /content/drive
Project: /content/TeachingMonster-released
Topics: /content/topics.xlsx


In [9]:
# 3) Install Python dependencies from the uploaded project.
# The project has pinned requirements; this may take several minutes on a fresh Colab runtime.

from pathlib import Path

project_dir = Path("/content/TeachingMonster-released")
requirements = project_dir / "requirements.txt"

if not requirements.exists():
    raise FileNotFoundError(f"requirements.txt not found: {requirements}")

%cd /content/TeachingMonster-released
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q openpyxl edge-tts bitsandbytes accelerate

print("Python dependencies installed.")

/content/TeachingMonster-released
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.6 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.48.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 2.3.0 requires fastapi<1,>=0.133, but you have fastapi 0.129.0 which i

In [10]:
# 4) GPU smoke test for the original cursor/vision modules.
# Run this after installing dependencies and before launching the batch.

import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, switch Runtime type to GPU and rerun setup cells.")

print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

try:
    import bitsandbytes as bnb
    print("bitsandbytes OK:", getattr(bnb, "__version__", "installed"))
except Exception as exc:
    raise RuntimeError(f"bitsandbytes import failed: {exc}") from exc

x = torch.randn(256, 256, device="cuda")
y = x @ x
print("CUDA matmul OK:", tuple(y.shape), y.dtype)

CUDA available: True
GPU count: 1
GPU name: NVIDIA A100-SXM4-40GB
CUDA version: 12.8
bitsandbytes OK: 0.49.2
CUDA matmul OK: (256, 256) torch.float32


In [11]:
# 5) Cursor model smoke test (original project path, GPU required).
# This loads the original QwenVL + UI-TARS cursor models once and shows GPU memory usage.
# Run this before the full batch if you want to verify model loading separately.

import os
import time
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

%cd /content/TeachingMonster-released

print("Before loading cursor models:")
!nvidia-smi
!du -sh ~/.cache/huggingface/hub 2>/dev/null || true

from src.cursor.cursor import CursorModule

start = time.time()
print("Loading CursorModule original QwenVL + UI-TARS models...", flush=True)
cursor_smoke = CursorModule(output_root="/content/cursor_smoke", final_root="/content/cursor_smoke_final")
cursor_smoke.load()
print(f"CursorModule loaded in {(time.time() - start) / 60:.1f} minutes", flush=True)

print("After loading cursor models:")
!nvidia-smi

# Free the smoke-test models before running the full batch, because the batch
# runs in a separate Python process and will load the original cursor models again.
import gc

del cursor_smoke
gc.collect()
torch.cuda.empty_cache()

print("After freeing smoke-test models:")
!nvidia-smi

/content/TeachingMonster-released
Before loading cursor models:
Mon Jul  6 11:32:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             53W /  400W |     462MiB /  40960MiB |      0%      Default |
|                                         |                        |        

JAX version 0.7.2, Flax version 0.11.2 available.


ImportError: cannot import name 'CUSTOM_KEY' from 'torch.ao.quantization' (/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/__init__.py)

In [12]:
# 4) Runtime configuration.
# Text LLM calls are forced to this OpenRouter model. If OpenRouter uses a different GPT 5.5 model id,
# change OPENROUTER_MODEL below before running the batch cell.

from pathlib import Path
from datetime import datetime
import os

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from colab_bootstrap import ensure_openai_key, setup_api_tracking, openai_model, print_api_usage

ensure_openai_key()
setup_api_tracking()
OPENAI_MODEL = openai_model()
OPENAI_IMAGE_MODEL = os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-1")
EDGE_TTS_VOICE = "en-US-GuyNeural"

PROJECT_DIR = Path("/content/TeachingMonster-released")
TOPICS_XLSX = Path("/content/topics.xlsx")
BATCH_OUTPUT_ROOT = Path("/content/teachingmonster_batch_outputs") / datetime.now().strftime("%Y%m%d_%H%M%S")
MAX_TOPICS = None  # None = all valid topics
FORCE_RERUN = False

os.environ["OPENAI_MODEL"] = OPENAI_MODEL
os.environ["OPENAI_IMAGE_MODEL"] = OPENAI_IMAGE_MODEL
os.environ["EDGE_TTS_VOICE"] = EDGE_TTS_VOICE

BATCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Model:", OPENAI_MODEL)
print("Project:", PROJECT_DIR)
print("Topics:", TOPICS_XLSX)
print("Output root:", BATCH_OUTPUT_ROOT)

Model: openai/gpt-5.5
Project: /content/TeachingMonster-released
Topics: /content/topics.xlsx
Output root: /content/teachingmonster_batch_outputs/20260706_113322


In [13]:
# 5) OpenRouter model call smoke test.
# This makes a real /chat/completions request using OPENROUTER_MODEL.
# If it fails, fix OPENROUTER_MODEL or the API key before running the full batch.

import httpx
import json

payload = {
    "model": OPENAI_MODEL,
    "messages": [{"role": "user", "content": "Reply with exactly: ok"}],
    "max_tokens": 32,
}
headers = {
    "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://github.com/teaching-monster",
    "X-OpenRouter-Title": "TeachingMonster",
}

response = httpx.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=60,
)

print("status:", response.status_code)
try:
    print(json.dumps(response.json(), ensure_ascii=False, indent=2)[:4000])
except Exception:
    print(response.text[:4000])

response.raise_for_status()
print("OpenAI model call test passed for:", OPENAI_MODEL)

status: 200
{
  "id": "gen-1783337606-7NXZrM1XMAVOAcuWBVPM",
  "object": "chat.completion",
  "created": 1783337606,
  "model": "openai/gpt-5.5-20260423",
  "provider": "OpenAI",
  "system_fingerprint": null,
  "service_tier": "default",
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "completed",
      "message": {
        "role": "assistant",
        "content": "ok",
        "refusal": null,
        "reasoning": null,
        "reasoning_details": [
          {
            "type": "reasoning.encrypted",
            "data": "gAAAAABqS5KH4-modm-dGbPtTYJV_UUztzyuNS2yoExXfjSmruie1smbPjEAVa-y-cjiAO0mr_KS9BPs_x5BoGs5g7OlvaS2yYP6uhQdu0gyvt-Ss5ktuGU8aLpCbHb5sVoZbVq8F1VM51aZmwDWUDDS7SyWqYuL6iUXT_qV6YH-GwLp_lvniuk_ZQCiy6cWpFhllFMsD97NcEJFMFxs3NQytqc_kLQmxGz00p_xCqG0vtwY-6aMJcKJUoKiV7LjElCJmXieRpF86WLnCOKvN-tfZ-T7ycT6Mqsbtgl-ZBrEW85LRy5Lu45u3YVJu--JH3ECUhDEM9aUW48whCFV9OUYVjOl__iaeEKableGJVhrn6ysVjKB3B5o52jyew5bvMlgtN9L2

In [14]:
# 6) Write the batch runner used by the final execution cell.

from pathlib import Path

runner_path = PROJECT_DIR / "scripts" / "batch_run_topics_colab.py"
runner_path.write_text(r'''
from __future__ import annotations

import argparse
import csv
import json
import os
import re
import shutil
import sys
import traceback
from contextlib import redirect_stderr, redirect_stdout
from datetime import datetime
from pathlib import Path
from typing import Any

import yaml
from openpyxl import load_workbook


def patch_openrouter_text_model(text_model: str, image_model: str) -> None:
    """Force all text LLM calls to the requested OpenRouter model at runtime."""
    import src.gemini_client as gemini_client
    import src.openrouter_client as openrouter_client

    original_resolve = openrouter_client.resolve_model

    def resolve_model(model: str | None, fallback: str) -> str:
        requested = (model or fallback or "").lower()
        if "image" in requested or "imagen" in requested:
            return image_model or original_resolve(model, fallback)
        return text_model

    def verbose_post(self, payload):
        url = f"{self.base_url}/chat/completions"
        import httpx
        with httpx.Client(timeout=300.0) as client:
            response = client.post(url, headers=self._headers(), json=payload)
            if response.status_code >= 400:
                print("OpenRouter request failed:", response.status_code, flush=True)
                print(response.text[:4000], flush=True)
            response.raise_for_status()
            return response.json()

    def generate_structured_without_response_format(
        self,
        prompt,
        response_schema,
        *,
        model=None,
        temperature=None,
        max_tokens=None,
    ):
        schema = response_schema.model_json_schema()
        structured_prompt = (
            f"{prompt}\n\n"
            "Return ONLY valid JSON. Do not include markdown fences, commentary, or extra text.\n"
            "The JSON must conform to this schema:\n"
            f"{json.dumps(schema, ensure_ascii=False)}"
        )
        text = self.chat(
            prompt=structured_prompt,
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            response_schema=None,
        )
        try:
            return response_schema.model_validate_json(text)
        except Exception:
            import json_repair

            cleaned = text.replace("```json", "").replace("```", "").strip()
            start = cleaned.find("{")
            end = cleaned.rfind("}") + 1
            if start != -1 and end > start:
                cleaned = cleaned[start:end]
            parsed = json_repair.loads(cleaned)
            return response_schema.model_validate(parsed)

    openrouter_client.resolve_model = resolve_model
    gemini_client.resolve_model = resolve_model
    openrouter_client.OpenRouterClient._post = verbose_post
    openrouter_client.OpenRouterClient.generate_structured = generate_structured_without_response_format


def patch_cursor_module_preserve_final_root() -> None:
    """Keep original cursor model logic, but do not delete the per-topic output folder."""
    import os
    import shutil

    from src.cursor.cursor import CursorModule

    def run(self, images, scripts, timestamps, audio_paths):
        if os.path.exists(self.output_root):
            shutil.rmtree(self.output_root)
        os.makedirs(f"{self.output_root}/slides", exist_ok=True)
        os.makedirs(f"{self.output_root}/videos", exist_ok=True)
        os.makedirs(self.final_root, exist_ok=True)

        self.images = images
        self.scripts = scripts
        self.timestamps = timestamps
        self.audio_paths = audio_paths

        grouped_scripts = self.grouping()
        grouped_points = self.grounding(grouped_scripts)
        return grouped_scripts, grouped_points

    CursorModule.run = run


def patch_tts_module_for_api(voice: str = "en-US-GuyNeural") -> None:
    """Replace GPU Qwen TTS with Edge online TTS plus word timing fallback."""
    import asyncio
    import wave

    import numpy as np

    from src.tts.tts import TTSModule

    def load(self):
        self.is_loaded = True
        self.voice = os.environ.get("EDGE_TTS_VOICE", voice)

    @staticmethod
    def ensure_monotonic(timings):
        out = []
        prev_end = 0.0
        for s, e in timings:
            s = max(float(s), prev_end, 0.0)
            e = max(float(e), s)
            out.append((s, e))
            prev_end = e
        return out

    def write_silence_wav(path: str, duration_sec: float = 0.5) -> None:
        sr = 22050
        n = max(1, int(duration_sec * sr))
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with wave.open(path, "w") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sr)
            wf.writeframes(np.zeros(n, dtype=np.int16).tobytes())

    def audio_duration(path: str, fallback: float) -> float:
        try:
            from moviepy.editor import AudioFileClip
            clip = AudioFileClip(path)
            duration = float(clip.duration)
            clip.close()
            return max(duration, 0.25)
        except Exception:
            return max(float(fallback), 0.25)

    def even_timings(script: str, total_duration: float):
        words = script.split()
        if not words:
            return []
        per = max(float(total_duration), 0.25) / len(words)
        return [(i * per, (i + 1) * per) for i in range(len(words))]

    async def synthesize(script: str, output_path: str, voice_name: str):
        import edge_tts

        boundary_timings = []
        audio_chunks = []
        communicate = edge_tts.Communicate(script, voice_name)
        async for chunk in communicate.stream():
            if chunk.get("type") == "audio" and chunk.get("data"):
                audio_chunks.append(chunk["data"])
            elif chunk.get("type") == "WordBoundary":
                raw_offset = chunk.get("offset", chunk.get("Offset", 0))
                raw_duration = chunk.get("duration", chunk.get("Duration", 0))
                offset = float(raw_offset or 0) / 10_000_000.0
                duration = float(raw_duration or 0) / 10_000_000.0
                if duration > 0:
                    boundary_timings.append((offset, offset + duration))

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "wb") as f:
            for data in audio_chunks:
                f.write(data)
        return boundary_timings

    def run(self, scripts):
        assert self.is_loaded, "Call load() before run()"
        os.makedirs(self.output_root, exist_ok=True)
        for fn in os.listdir(self.output_root):
            if fn.endswith(".mp3") and fn[:-4].isdigit():
                try:
                    os.remove(os.path.join(self.output_root, fn))
                except OSError:
                    pass

        all_timings = []
        for idx, script in enumerate(scripts, start=1):
            mp3_path = os.path.join(self.output_root, f"{idx}.mp3")
            script = script if isinstance(script, str) else ""
            words = script.split()

            if not words:
                write_silence_wav(mp3_path, 0.5)
                all_timings.append([])
                continue

            try:
                timings = asyncio.run(synthesize(script, mp3_path, self.voice))
                duration = audio_duration(mp3_path, fallback=max(3.0, 0.35 * len(words)))
                if len(timings) < len(words):
                    timings = even_timings(script, duration)
                else:
                    timings = timings[: len(words)]
                all_timings.append(ensure_monotonic(timings))
            except Exception:
                duration = max(3.0, 0.35 * len(words))
                write_silence_wav(mp3_path, duration)
                all_timings.append(ensure_monotonic(even_timings(script, duration)))

        return self.output_root, all_timings

    TTSModule.load = load
    TTSModule.run = run


def patch_cursor_module_for_cpu() -> None:
    """Replace GPU vision cursor planning with deterministic slide focus points."""
    from src.cursor.cursor import CursorModule

    def load(self):
        self.is_loaded = True

    def run(self, images, scripts, timestamps, audio_paths):
        assert self.is_loaded, "Call load() before run()"
        grouped_scripts = []
        grouped_points = []

        for image, script in zip(images, scripts):
            width, height = image.size
            script = script if isinstance(script, str) and script.strip() else "placeholder"

            # Split long narration into a few chunks so the pointer still moves naturally.
            sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", script) if s.strip()]
            if not sentences:
                sentences = [script]
            if len(sentences) > 4:
                chunk_size = max(1, (len(sentences) + 3) // 4)
                chunks = [" ".join(sentences[i : i + chunk_size]) for i in range(0, len(sentences), chunk_size)]
            else:
                chunks = sentences

            focus_points = [
                (int(width * 0.50), int(height * 0.50)),
                (int(width * 0.28), int(height * 0.38)),
                (int(width * 0.72), int(height * 0.38)),
                (int(width * 0.50), int(height * 0.70)),
            ]
            points = [focus_points[i % len(focus_points)] for i in range(len(chunks))]

            grouped_scripts.append(chunks)
            grouped_points.append(points)

        return grouped_scripts, grouped_points

    CursorModule.load = load
    CursorModule.run = run


def slugify(value: str, max_len: int = 90) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value).strip())
    value = re.sub(r"_+", "_", value).strip("._-")
    return (value or "topic")[:max_len]


def clean(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def load_topic_rows(xlsx_path: Path) -> list[dict[str, str]]:
    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    rows = ws.iter_rows(values_only=True)
    header = [clean(v) for v in next(rows)]

    required = {"Request ID", "Topic", "Student Persona"}
    missing = required - set(header)
    if missing:
        raise ValueError(f"Missing required columns in first sheet {ws.title!r}: {sorted(missing)}")

    out: list[dict[str, str]] = []
    for row_idx, raw_row in enumerate(rows, start=2):
        item = {header[i]: clean(raw_row[i]) if i < len(raw_row) else "" for i in range(len(header))}
        request_id = item.get("Request ID", "")
        topic = item.get("Topic", "")
        persona = item.get("Student Persona", "")

        # Skip section/group rows such as "1. Computer Science" and repeated header rows.
        if not request_id or not topic or not persona:
            continue
        if request_id == "Request ID" or topic == "Topic":
            continue

        item["_row_number"] = str(row_idx)
        out.append(item)

    return out


def build_prompts(row: dict[str, str]) -> tuple[str, str]:
    bloom_taxonomy = row.get("Bloom's Taxonomy", "")
    requirement_parts = [
        f"Request ID: {row.get('Request ID', '')}",
        f"Difficulty Level: {row.get('Difficulty Level', '')}",
        f"Big Concept: {row.get('Big Concept', '')}",
        f"Topic: {row.get('Topic', '')}",
        f"Bloom's Taxonomy: {bloom_taxonomy}",
        f"ICAP Level: {row.get('ICAP Level', '')}",
        "Learning Goal:",
        row.get("Learning Goal", ""),
        "Key Learning Points:",
        row.get("Key Learning Points", ""),
        "Instruction: Generate one educational video that follows these metadata and learning goals closely.",
    ]
    requirement_prompt = "\n\n".join(part for part in requirement_parts if clean(part))
    persona_prompt = row.get("Student Persona", "")
    return requirement_prompt, persona_prompt


def make_config(project_dir: Path, run_dir: Path, text_model: str, image_model: str, api_key: str):
    from src.config_schema import AppConfig

    config_path = project_dir / "config" / "default.yaml"
    with config_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    data.setdefault("llm", {})
    data["llm"]["provider"] = "openrouter"
    data["llm"]["base_url"] = "https://api.openai.com/v1"
    data["llm"]["api_key"] = os.environ["OPENAI_API_KEY"]
    data["llm"]["default_model"] = text_model
    data["llm"]["default_image_model"] = image_model
    data["llm"]["api_key"] = api_key

    data.setdefault("output", {})
    data["output"]["tmp_dir"] = str(run_dir / "tmp_pipeline")
    data["output"]["final_video_dir"] = str(run_dir)

    return AppConfig(**data)


def copy_debug_json(project_dir: Path, run_dir: Path) -> None:
    src_tmp = project_dir / "tmp"
    dst = run_dir / "debug_json"
    dst.mkdir(parents=True, exist_ok=True)
    for name in ["outlines.json", "slides_struct.json", "scripts.json", "word_timings.json"]:
        src = src_tmp / name
        if src.exists():
            shutil.copy2(src, dst / name)


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


class Tee:
    """Write redirected output to both the notebook stream and the per-topic log."""

    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            try:
                stream.write(data)
                stream.flush()
            except Exception:
                pass

    def flush(self):
        for stream in self.streams:
            try:
                stream.flush()
            except Exception:
                pass


def run_batch(args: argparse.Namespace) -> None:
    project_dir = Path(args.project_dir).resolve()
    topics_xlsx = Path(args.topics_xlsx).resolve()
    output_root = Path(args.output_root).resolve()
    text_model = args.model
    image_model = args.image_model
    api_key = args.api_key or os.environ.get("OPENAI_API_KEY", "")

    if not api_key:
        raise ValueError("OPENAI_API_KEY is required")
    if not project_dir.exists():
        raise FileNotFoundError(project_dir)
    if not topics_xlsx.exists():
        raise FileNotFoundError(topics_xlsx)

    os.environ["OPENAI_API_KEY"] = api_key
    os.environ["OPENAI_MODEL"] = text_model
    os.environ["OPENAI_IMAGE_MODEL"] = image_model

    sys.path.insert(0, str(project_dir))
    os.chdir(project_dir)
    (project_dir / "tmp").mkdir(exist_ok=True)
    output_root.mkdir(parents=True, exist_ok=True)

    patch_openrouter_text_model(text_model, image_model)
    patch_cursor_module_preserve_final_root()
    patch_tts_module_for_api()

    from scripts.T2V_pipeline import VideoGenerationPipeline
    from src.gemini_client import GeminiClient

    rows = load_topic_rows(topics_xlsx)
    if args.max_topics is not None:
        rows = rows[: args.max_topics]
    write_json(output_root / "input_rows.json", rows)

    manifest_path = output_root / "manifest.csv"
    fieldnames = ["row_number", "request_id", "topic", "status", "run_dir", "final_video_path", "started_at", "ended_at", "error"]

    with manifest_path.open("w", newline="", encoding="utf-8") as mf:
        writer = csv.DictWriter(mf, fieldnames=fieldnames)
        writer.writeheader()

    print(f"Loaded {len(rows)} valid topic rows from {topics_xlsx}")
    print(f"Writing outputs to {output_root}")

    cached_cursor_module = None

    def load_pipeline_modules_with_cached_cursor(pipeline):
        nonlocal cached_cursor_module
        pipeline.outline_module.load()
        pipeline.wrapper.load()
        pipeline.slides_module.load()
        pipeline.tts_module.load()

        if cached_cursor_module is None:
            print("    Loading original cursor models once (QwenVL + UI-TARS)...", flush=True)
            pipeline.cursor_module.load()
            cached_cursor_module = pipeline.cursor_module
        else:
            print("    Reusing cached cursor models.", flush=True)
            cached_cursor_module.output_root = pipeline.cursor_output_root
            cached_cursor_module.final_root = os.path.dirname(pipeline.video_output_path)
            cached_cursor_module.final_video_name = os.path.basename(pipeline.video_output_path)
            cached_cursor_module.is_loaded = True
            pipeline.cursor_module = cached_cursor_module

    for idx, row in enumerate(rows, start=1):
        request_id = row["Request ID"]
        topic = row["Topic"]
        run_dir = output_root / f"{idx:03d}_{slugify(request_id)}"
        run_dir.mkdir(parents=True, exist_ok=True)
        final_video = run_dir / f"{slugify(request_id)}.mp4"
        log_path = run_dir / "run.log"
        started_at = datetime.utcnow().isoformat() + "Z"
        status = "success"
        error = ""

        print(f"[{idx}/{len(rows)}] {request_id}: {topic}")

        if final_video.exists() and not args.force:
            status = "skipped_existing"
            ended_at = datetime.utcnow().isoformat() + "Z"
        else:
            try:
                requirement_prompt, persona_prompt = build_prompts(row)
                write_json(run_dir / "input_row.json", row)
                (run_dir / "requirement_prompt.txt").write_text(requirement_prompt, encoding="utf-8")
                (run_dir / "persona_prompt.txt").write_text(persona_prompt, encoding="utf-8")

                config = make_config(project_dir, run_dir, text_model, image_model, api_key)
                client = GeminiClient(config.llm)
                pipeline = VideoGenerationPipeline(
                    llm_client=client,
                    app_config=config,
                    output_video_name=final_video.name,
                    final_video_dir=str(run_dir),
                )

                print(f"  Loading pipeline modules for {request_id}...", flush=True)
                with log_path.open("w", encoding="utf-8") as log_file:
                    with redirect_stdout(Tee(sys.stdout, log_file)), redirect_stderr(Tee(sys.stderr, log_file)):
                        load_pipeline_modules_with_cached_cursor(pipeline)
                print(f"  Pipeline modules loaded for {request_id}.", flush=True)

                print(f"  Running generation for {request_id}...", flush=True)
                with log_path.open("a", encoding="utf-8") as log_file:
                    with redirect_stdout(Tee(sys.stdout, log_file)), redirect_stderr(Tee(sys.stderr, log_file)):
                        assets = pipeline.run(
                            requirement_prompt=requirement_prompt,
                            persona_prompt=persona_prompt,
                        )
                        write_json(run_dir / "assets.json", assets)
                print(f"  Finished generation for {request_id}.", flush=True)

                copy_debug_json(project_dir, run_dir)
                ended_at = datetime.utcnow().isoformat() + "Z"
            except Exception as exc:
                status = "failed"
                error = f"{type(exc).__name__}: {exc}"
                ended_at = datetime.utcnow().isoformat() + "Z"
                with log_path.open("a", encoding="utf-8") as log_file:
                    log_file.write("\n\n=== BATCH RUNNER EXCEPTION ===\n")
                    traceback.print_exc(file=log_file)
                print(f"  FAILED: {error}")

        with manifest_path.open("a", newline="", encoding="utf-8") as mf:
            writer = csv.DictWriter(mf, fieldnames=fieldnames)
            writer.writerow({
                "row_number": row.get("_row_number", ""),
                "request_id": request_id,
                "topic": topic,
                "status": status,
                "run_dir": str(run_dir),
                "final_video_path": str(final_video if final_video.exists() else ""),
                "started_at": started_at,
                "ended_at": ended_at,
                "error": error,
            })

    print("Batch complete.")
    print("Manifest:", manifest_path)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run TeachingMonster for every valid row in topics.xlsx")
    parser.add_argument("--project-dir", required=True)
    parser.add_argument("--topics-xlsx", required=True)
    parser.add_argument("--output-root", required=True)
    parser.add_argument("--api-key", default="")
    parser.add_argument("--model", required=True)
    parser.add_argument("--image-model", default="google/gemini-2.5-flash-image")
    parser.add_argument("--max-topics", type=int, default=None)
    parser.add_argument("--force", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    run_batch(parse_args())
''', encoding="utf-8")

print("Wrote batch runner:", runner_path)

Wrote batch runner: /content/TeachingMonster-released/scripts/batch_run_topics_colab.py


In [ ]:
# 7) Run all topics with live subprocess logs and GPU monitoring.
# This can take a long time because each row generates a full video. Keep the Colab runtime connected.

import os
import subprocess
import sys
import threading
import time
from pathlib import Path

cmd = [
    sys.executable,
    "-u",
    str(PROJECT_DIR / "scripts" / "batch_run_topics_colab.py"),
    "--project-dir", str(PROJECT_DIR),
    "--topics-xlsx", str(TOPICS_XLSX),
    "--output-root", str(BATCH_OUTPUT_ROOT),
    "--api-key", os.environ["OPENAI_API_KEY"],
    "--model", OPENAI_MODEL,
    "--image-model", OPENAI_IMAGE_MODEL,
]

if MAX_TOPICS is not None:
    cmd.extend(["--max-topics", str(MAX_TOPICS)])

if FORCE_RERUN:
    cmd.append("--force")

env = os.environ.copy()
env["EDGE_TTS_VOICE"] = EDGE_TTS_VOICE
env["PYTHONUNBUFFERED"] = "1"
env["TRANSFORMERS_VERBOSITY"] = "info"

stop_monitor = threading.Event()

def monitor_gpu(interval_sec: int = 60):
    while not stop_monitor.is_set():
        print("\n--- GPU monitor ---", flush=True)
        subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw",
                "--format=csv,noheader,nounits",
            ],
            check=False,
        )
        subprocess.run(["bash", "-lc", "du -sh ~/.cache/huggingface/hub 2>/dev/null || true"], check=False)
        stop_monitor.wait(interval_sec)

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()

print("Running:", " ".join(cmd[:7]), "...", flush=True)
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
finally:
    stop_monitor.set()
    monitor_thread.join(timeout=5)

print("Done. Output root:", BATCH_OUTPUT_ROOT)


In [ ]:
!nvidia-smi
!du -sh ~/.cache/huggingface/hub 2>/dev/null || true

Sun Jul  5 17:02:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             48W /  400W |     462MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# 8) Zip latest outputs for download.
# Works even after a kernel restart by finding the newest /content output folder.

from pathlib import Path
from google.colab import files
import shutil


def resolve_batch_output_root() -> Path:
    if "BATCH_OUTPUT_ROOT" in globals():
        p = Path(BATCH_OUTPUT_ROOT)
        if p.exists():
            return p

    base = Path("/content/teachingmonster_batch_outputs")
    candidates = [p for p in base.iterdir() if p.is_dir()] if base.exists() else []
    if not candidates:
        raise FileNotFoundError("No output folders found under /content/teachingmonster_batch_outputs")
    return max(candidates, key=lambda p: p.stat().st_mtime)


src_dir = resolve_batch_output_root()
zip_path = shutil.make_archive(str(src_dir), "zip", root_dir=src_dir)
print("Output folder:", src_dir)
print("Created:", zip_path)
files.download(zip_path)

Output folder: /content/teachingmonster_batch_outputs/20260706_113322
Created: /content/teachingmonster_batch_outputs/20260706_113322.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# 9) Copy latest outputs to Google Drive for persistent storage.
# Works even after a kernel restart by finding the newest /content output folder.
# Destination:
#   /content/drive/MyDrive/education_video/outputs/<batch_name>/
#   /content/drive/MyDrive/education_video/outputs/<batch_name>.zip

from pathlib import Path
import os
import shutil
import sys


def resolve_batch_output_root() -> Path:
    if "BATCH_OUTPUT_ROOT" in globals():
        p = Path(BATCH_OUTPUT_ROOT)
        if p.exists():
            return p

    base = Path("/content/teachingmonster_batch_outputs")
    candidates = [p for p in base.iterdir() if p.is_dir()] if base.exists() else []
    if not candidates:
        raise FileNotFoundError("No output folders found under /content/teachingmonster_batch_outputs")
    return max(candidates, key=lambda p: p.stat().st_mtime)


in_colab = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if in_colab:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Google Drive is not mounted at /content/drive/MyDrive")

src_dir = resolve_batch_output_root()
outputs_dir = Path("/content/drive/MyDrive/education_video/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

dst_dir = outputs_dir / src_dir.name
if dst_dir.exists():
    shutil.rmtree(dst_dir)
shutil.copytree(src_dir, dst_dir)

src_zip = src_dir.with_suffix(".zip")
if not src_zip.exists():
    src_zip = Path(shutil.make_archive(str(src_dir), "zip", root_dir=src_dir))

dst_zip = outputs_dir / src_zip.name
shutil.copy2(src_zip, dst_zip)

print("Output folder:", src_dir)
print("Copied output folder to:", dst_dir)
print("Copied zip to:", dst_zip)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output folder: /content/teachingmonster_batch_outputs/20260706_113322
Copied output folder to: /content/drive/MyDrive/education_video/outputs/20260706_113322
Copied zip to: /content/drive/MyDrive/education_video/outputs/20260706_113322.zip
